# INet-Tool Pipeline Walkthrough
## From Raw Data to Consensus Network

This notebook demonstrates the complete INet pipeline on 3 small synthetic datasets.

**Algorithm**: INet (Policastro et al. 2024)  
**Implementation**: Python `inet_tool` (translated from R-INetTool)

Each cell shows exact matrix outputs so you can trace every computation.


## 1. Setup

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\Faruk\Desktop\VSCode Projeler\INet-Tool-Python")
import numpy as np
import pandas as pd
import networkx as nx
import inet_tool
from IPython.display import display

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.precision', 4)
print("inet_tool loaded. Version:", inet_tool.__version__)


## 2. Create Raw Data (3 layers × 5 patients × 4 features)

We create 3 small datasets simulating different omics measurements for **5 patients**:
- **Layer 1** (e.g., Gene Expression): 4 genes × 5 patients
- **Layer 2** (e.g., Methylation): 4 probes × 5 patients  
- **Layer 3** (e.g., miRNA): 4 miRNAs × 5 patients

**Row = feature (gene/probe/miRNA), Column = patient.**

Patient P1 and P2 are similar across layers. P4 and P5 are similar. P3 is an outlier.


In [ ]:
np.random.seed(42)

# Layer 1: 4 features x 5 patients
data1 = np.array([
    [ 2.0,  1.8, -0.5, -1.2, -1.5],   # Gene A: high in P1,P2
    [ 1.5,  1.3,  0.1, -1.0, -1.3],   # Gene B: high in P1,P2
    [-0.3, -0.2,  1.5,  2.0,  1.8],   # Gene C: high in P3,P4,P5
    [ 0.1,  0.0,  2.0,  2.5,  2.2],   # Gene D: high in P3,P4,P5
])

# Layer 2: different features, same patients
data2 = np.array([
    [ 1.8,  2.0, -0.3, -1.5, -1.8],   # Probe 1
    [ 1.2,  1.1,  0.3, -0.8, -1.0],   # Probe 2
    [-0.5, -0.4,  1.8,  2.2,  1.9],   # Probe 3
    [ 0.0, -0.1,  2.2,  2.8,  2.5],   # Probe 4
])

# Layer 3: different features again
data3 = np.array([
    [ 2.2,  2.1, -0.8, -1.0, -1.2],   # miRNA A
    [ 1.4,  1.5,  0.0, -1.2, -1.4],   # miRNA B
    [-0.1, -0.3,  1.2,  1.8,  2.0],   # miRNA C
    [ 0.3,  0.1,  1.9,  2.3,  2.0],   # miRNA D
])

data_list = [data1, data2, data3]

for i, d in enumerate(data_list):
    df = pd.DataFrame(d, 
        index=[f"Feature_{j+1}" for j in range(d.shape[0])],
        columns=[f"P{j+1}" for j in range(d.shape[1])])
    print(f"=== Layer {i+1} ({d.shape[0]} features x {d.shape[1]} patients) ===")
    display(df)
    print()


## 3. `construction_graph` — Build Similarity Networks

This function:
1. **Computes Pearson correlation** between patients (columns)
2. **Takes the 80th percentile** as threshold (we use `perc=0.80` to keep more edges and see more interesting structure than the default 0.95)
3. **Zeroes correlations below threshold**
4. **Builds a weighted graph** (edges = correlations above threshold)

**Why perc=0.80?** With 5 patients, the 95th percentile leaves almost no edges. Using 0.80 keeps ~1-2 edges so we can see the algorithm in action.


In [ ]:
# Use perc=0.85 to keep top 15% of correlations
cg = inet_tool.construction_graph(data_list, perc=0.85, plot=False)

for i in range(3):
    ti = cg["Threshold"][i]
    adj = cg["Adj"][i]
    df_adj = pd.DataFrame(adj, 
        columns=[f"P{j+1}" for j in range(adj.shape[0])],
        index=[f"P{j+1}" for j in range(adj.shape[0])])
    print(f"=== Layer {i+1} Adjacency Matrix ===")
    print(f"  Threshold (85th pct): {ti.get('0.85', 'N/A'):.4f}")
    print(f"  Edges: {ti.get('edge', 'N/A')}")
    display(df_adj)
    print()


## 4. Pre-Analysis — Weighted Jaccard Distance

Before running the algorithm, we measure how **dissimilar** the 3 layers are.

The **weighted Jaccard distance** between two graphs:
- Extracts all edge weights into vectors
- Computes: `sim(a,b) = sum(min(a_i, b_i)) / sum(max(a_i, b_i))` 
- Distance = `1 - sim`
- 0 = identical graphs, 1 = completely different


In [ ]:
# Full distance matrix
jw_dist = inet_tool.jw_matrix(cg["Graphs"])
jw_df = pd.DataFrame(jw_dist, 
    columns=["Layer 1","Layer 2","Layer 3"],
    index=["Layer 1","Layer 2","Layer 3"])
print("Weighted Jaccard Distance Matrix (0=identical, 1=completely different):")
display(jw_df)
print()

# Mean distance
jw_mean_val = inet_tool.jw_mean(cg["Graphs"])
print(f"Mean Jaccard distance: {jw_mean_val:.6f}")
print()
print(f"(This means the layers are {'quite different' if jw_mean_val > 0.5 else 'fairly similar'} before the algorithm starts)")


## 5. `consensus_net` — THE CORE ALGORITHM

This is the main event. The algorithm iteratively modifies edge weights to bring the layers closer together.

We use:
- `threshold=0.3` — lower threshold to keep more consensus edges
- `tolerance=0.3` — stop when Jaccard distance drops below 0.3
- `theta=0.04` — neighborhood influence weight (default)
- `verbose=True` — show each iteration


In [ ]:
# Run with verbose=True to see iterations
con = inet_tool.consensus_net(
    cg["Adj"], 
    threshold=0.3, 
    tolerance=0.3, 
    theta=0.04, 
    nitermax=50, 
    ncores=1,
    verbose=True
)

print()
print("========================================")
print("CONSENSUS OUTPUT")
print("========================================")
print(f"Iterations completed: {len(con['Comparison']) - 1}")
print(f"Comparison (distance per iteration): {con['Comparison']}")
print()

# Show consensus adjacency
cons_adj = nx.to_numpy_array(con["graphConsensus"], weight="weight")
df_cons = pd.DataFrame(cons_adj,
    columns=[f"P{j+1}" for j in range(cons_adj.shape[0])],
    index=[f"P{j+1}" for j in range(cons_adj.shape[0])])
print(f"Consensus Network: {con['graphConsensus'].number_of_nodes()} nodes, {con['graphConsensus'].number_of_edges()} edges")
display(df_cons)


## 6. Inside the Algorithm — One Weight Computation, Step by Step

Here we manually trace what happens **inside** the consensus loop for one specific edge. This reveals exactly how the weight update formula works.

We pick edge `(0,1)` — patients P1 and P2 — and trace through layers.


In [ ]:
# Rebuild the internal structures the algorithm uses
from inet_tool._internal import code, weight_mat, get_lower_tri_noDiag
from inet_tool.consensus import _adj_to_graphs, _build_hashes
from inet_tool.distance import jaccard_all

graphs = _adj_to_graphs(cg["Adj"])
K = len(graphs)
N = graphs[0].number_of_nodes()

# Build the union of all edges
union_edges = set()
for g in graphs:
    for u, v in g.edges():
        union_edges.add((min(u, v), max(u, v)))
edgelist = sorted(union_edges)

# Build hashmaps (neighbors + weights)
neig_list, weights_list = _build_hashes(graphs, edgelist)

print("=" * 60)
print(f"Graphs: {K} layers, {N} nodes each")
print(f"Union edges: {edgelist}")
print()

# Trace edge (0,1)
u, v = 0, 1
print(f"TRACING EDGE ({u},{v}) — Patients P1 and P2")
print("=" * 60)

for i in range(K):
    print(f"
--- LAYER {i+1} ---")
    
    # Neighbors
    inei = neig_list[i][u]
    jnei = neig_list[i][v]
    inter = inei & jnei
    print(f"  Neighbors of P{u+1}: {inei}")
    print(f"  Neighbors of P{v+1}: {jnei}")
    print(f"  Common neighbors: {inter}")
    
    # Edge weight in this layer
    w_uso = weights_list[i].get(code(u, v), 0.0)
    print(f"  Edge weight in this layer: {w_uso:.4f}")
    
    # Edge weights in other layers
    w_others = []
    for j in range(K):
        if j == i:
            continue
        w_altri = weights_list[j].get(code(u, v), 0.0)
        w_others.append(w_altri)
        print(f"  Edge weight in layer {j+1}: {w_altri:.4f}")
    
    # Base weight
    peso = (w_uso + np.mean(w_others)) / 2.0
    print(f"
  1) BASE WEIGHT (peso):")
    print(f"     = (w_uso + mean(w_others)) / 2")
    print(f"     = ({w_uso:.4f} + {np.mean(w_others):.4f}) / 2")
    print(f"     = {peso:.6f}")
    
    # Ego-network weight (simplified computation)
    if len(inter) == 0:
        pesi_ego = 0.0
        print(f"
  2) EGO WEIGHT: 0 (no common neighbors)")
    else:
        # For each common neighbor, compute contribution
        all_intersections = []
        for j in range(K):
            all_intersections.extend(neig_list[j][u] & neig_list[j][v])
        from collections import Counter
        counter = Counter(all_intersections)
        
        pint_sum = 0.0
        print(f"
  2) EGO WEIGHT computation:")
        for k in inter:
            p1 = weights_list[i].get(code(u, k), 0.0)
            p2 = weights_list[i].get(code(v, k), 0.0)
            ncom = counter[k]
            contribution = (ncom / K) * (p1 + p2)
            print(f"     Neighbor P{k+1}: w(u,k)={p1:.4f}, w(v,k)={p2:.4f}, "
                  f"appears_in={ncom}/{K} layers, contrib={(ncom/K)*(p1+p2):.6f}")
            pint_sum += contribution
        
        denom = len(inei) + len(jnei) - (2 if w_uso != 0 else 0)
        n_root = len(inter)
        pesi_ego = (pint_sum / denom) ** (1.0 / n_root) if denom > 0 else 0.0
        
        print(f"
     Sum of contributions: {pint_sum:.6f}")
        print(f"     Denominator (|N(u)|+|N(v)|-{'2' if w_uso!=0 else '0'}): {denom}")
        print(f"     Ego weight = ({pint_sum:.6f}/{denom})^(1/{n_root}) = {pesi_ego:.6f}")
    
    # Compute ego for other layers
    pesi_ego_others = []
    for j in range(K):
        if j == i:
            continue
        inter_j = neig_list[j][u] & neig_list[j][v]
        if len(inter_j) == 0:
            pesi_ego_others.append(0.0)
        else:
            inter_other, denom_other, pint_other = 0, 0, 0  # simplified
            # (same computation as above but for layer j)
            all_int = []
            for h in range(K):
                all_int.extend(neig_list[h][u] & neig_list[h][v])
            ctr = Counter(all_int)
            ps = 0.0
            for k in inter_j:
                p1 = weights_list[j].get(code(u, k), 0.0)
                p2 = weights_list[j].get(code(v, k), 0.0)
                ps += (ctr[k] / K) * (p1 + p2)
            d = len(neig_list[j][u]) + len(neig_list[j][v]) - (2 if weights_list[j].get(code(u,v),0)!=0 else 0)
            ego_other = (ps / d) ** (1.0/len(inter_j)) if d > 0 else 0.0
            pesi_ego_others.append(ego_other)
    
    mean_ego_others = np.mean(pesi_ego_others) if pesi_ego_others else 0.0
    
    print(f"
  3) EGO WEIGHTS in other layers: {[f'{x:.6f}' for x in pesi_ego_others]}")
    print(f"     Mean ego-others: {mean_ego_others:.6f}")
    
    # Final ego weight
    pesi_ego_final = (pesi_ego + mean_ego_others) / 2.0
    print(f"
  4) FINAL EGO WEIGHT:")
    print(f"     = (ego_own + mean(ego_others)) / 2")
    print(f"     = ({pesi_ego:.6f} + {mean_ego_others:.6f}) / 2")
    print(f"     = {pesi_ego_final:.6f}")
    
    # Final weight
    weight_new = peso + 0.04 * pesi_ego_final
    weight_new = min(weight_new, 1.0)
    print(f"
  5) FINAL WEIGHT (theta=0.04):")
    print(f"     = peso + theta * ego_weight")
    print(f"     = {peso:.6f} + 0.04 * {pesi_ego_final:.6f}")
    print(f"     = {peso + 0.04*pesi_ego_final:.6f}")
    if peso + 0.04 * pesi_ego_final > 1.0:
        print(f"     → CLAMPED to 1.0")
    print(f"     = {weight_new:.6f}")


## 7. `density_net` — Weight Distribution After Consensus

Shows the distribution of average edge weights across layers. Useful for choosing a good consensus threshold.


In [ ]:
dens = inet_tool.density_net(con["similarGraphs"])
print("Quantiles of mean weights (all values):")
print(dens["quantile"])
print()
print("Quantiles of mean weights (excluding zeros):")
print(dens["quantileNo0"])


## 8. `threshold_net` — Re-Threshold the Consensus

Without re-running the algorithm, we can try different thresholds on the similar graphs to see how the consensus changes.


In [ ]:
for t in [0.2, 0.3, 0.5]:
    th = inet_tool.threshold_net(con["similarGraphs"], threshold=t)
    adj_th = nx.to_numpy_array(th, weight="weight")
    print(f"threshold={t}: {th.number_of_edges()} edges")
    if th.number_of_edges() > 0:
        df_th = pd.DataFrame(adj_th,
            columns=[f"P{j+1}" for j in range(adj_th.shape[0])],
            index=[f"P{j+1}" for j in range(adj_th.shape[0])])
        display(df_th)
    print()


## 9. `specific_net` — Layer-Specific Edges

Shows which edges are **unique to each layer** (not present in the consensus). High specificity means the layer captures information the others don't.


In [ ]:
spec = inet_tool.specific_net(cg["Graphs"], con["graphConsensus"])
for i in range(3):
    g_spec = spec["GraphsDifference"][i]
    pct = spec["percentageOfSpecificity"][i] * 100
    print(f"Layer {i+1}: {g_spec.number_of_edges()} specific edges ({pct:.1f}%)")
    if g_spec.number_of_edges() > 0:
        adj_spec = nx.to_numpy_array(g_spec, weight="weight")
        df_spec = pd.DataFrame(adj_spec,
            columns=[f"P{j+1}" for j in range(adj_spec.shape[0])],
            index=[f"P{j+1}" for j in range(adj_spec.shape[0])])
        display(df_spec)
    print()


## 10. `measures_net` — Network Statistics

Graph-level and node-level measures for each similar graph.


In [ ]:
meas = inet_tool.measures_net(con["similarGraphs"], nodes_measures=False)
measure_names = ["vertices","edges","transitivity","diameter","modularityLouvain",
                 "edgeDensity","assortativity","centrDegree","centrBetween"]
for i in range(3):
    gm = meas[i]["graphsMeasures"].flatten()
    df_meas = pd.DataFrame(gm, index=measure_names, columns=["value"])
    print(f"=== Layer {i+1} ===")
    display(df_meas)
    print()


## 11. `plot_c` — Visualize Consensus

Plot the consensus network minus isolated nodes.


In [ ]:
print("Consensus network structure:")
g = con["graphConsensus"]
for u, v, data in g.edges(data="weight"):
    print(f"  P{u+1} --- P{v+1}  (weight={data:.6f})")
print()

# Show isolated nodes
deg = dict(g.degree())
isolated = [v+1 for v, d in deg.items() if d == 0]
if isolated:
    print(f"Isolated patients (no consensus edges): {isolated}")

inet_tool.plot_c(g, vertex_size=300, vertex_label_size=12)


## 12. Pipeline Summary

```
Raw Data (3 layers, features x patients)
    │
    ▼ construction_graph(perc=0.85)
Adjacency Matrices + Graphs (patient-patient correlation)
    │
    ├── jw_matrix / jw_mean   →  "How different are the layers?"
    │
    ▼ consensus_net(tolerance=0.3, theta=0.04)
       │
       │  WHILE distance > tolerance:
       │    ├── Compute Jaccard distance
       │    ├── Build union of all edges
       │    ├── For each layer:
       │    │   └── For each edge in union:
       │    │       peso = (w_own + mean(w_others)) / 2
       │    │       ego  = neighborhood structural weight
       │    │       new  = min(peso + theta*ego, 1.0)
       │    │
       │    └── Accept if distance decreased
       │
       └── Element-wise mean → threshold → Consensus Graph
    │
    ├── density_net    →  Weight distribution
    ├── threshold_net  →  Try different thresholds
    ├── specific_net   →  Layer-unique edges
    ├── measures_net   →  Graph statistics
    └── plot_c         →  Visualization
```
